# Data Understanding & Preparation - Heart Diagnoses

## Overview

This notebook documents the complete pipeline for extracting clinical features from the **Heart Diagnoses** table and aggregating them at the **patient level** to create comprehensive patient profiles for subsequent clustering analysis.


## Setup & Libraries

In [1]:
import pandas as pd
import numpy as np
import re

# Data Understanding

## Dataset Overview and Initial Assessment

The Heart Diagnoses table contains **clinical notes and structured data** from hospitalized cardiac patients. The data is linked by:
- `hadm_id`: Hospital admission identifier (unique per admission)
- `subject_id`: Patient identifier (unique per patient, multiple admissions possible)

### Key Data Characteristics
- **Multiple admissions per patient**: Longitudinal temporal data
- **Mixed data types**: Numeric (vital signs), categorical (gender, diagnoses), unstructured text (clinical notes)
- **Clinical domains**: Physical examination, echocardiography, cardiac catheterization, ECG, CT imaging

### Data Quality Dimensions Assessed
1. **Dimensionality**: Number of admissions and data fields
2. **Duplicate Records**: Exact row duplicates and key-based redundancy
3. **Missing Values**: Percentage of NaN or empty fields per column
4. **Data Types**: Proper encoding of temporal (datetime) and numeric fields
5. **Value Distributions**: Summary statistics for key variables

In [2]:
# Load data
from pathlib import Path


df = pd.read_csv(Path().resolve().parent / 'Data' / 'heart_diagnoses_1.csv')

print("="*70)
print("INITIAL DATA ASSESSMENT")
print("="*70)
print(f"\nDataset shape: {df.shape[0]} admissions × {df.shape[1]} columns")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nFirst few rows:")
print(df.head(3))

INITIAL DATA ASSESSMENT

Dataset shape: 4864 admissions × 25 columns

Column names and types:
note_id             object
subject_id           int64
hadm_id              int64
note_type           object
note_seq             int64
charttime           object
storetime           object
HPI                 object
physical_exam       object
chief_complaint     object
invasions           object
X-ray               object
CT                  object
Ultrasound          object
CATH                object
ECG                 object
MRI                 object
reports             object
subject_id_dx        int64
icd_code            object
long_title          object
gender              object
age                float64
anchor_year        float64
dod                 object
dtype: object

First few rows:
          note_id  subject_id   hadm_id note_type  note_seq  \
0  10000980-DS-20    10000980  29654838        DS        20   
1  10000980-DS-21    10000980  26913865        DS        21   
2   1000201

## Column Selection & Data Quality

We drop non-informative or technical columns because we don't require them for the creation of the patient profiles.

In [3]:
columns_to_drop = ["note_type", "note_seq", "subject_id_dx", "anchor_year", "note_id"]
df = df.drop(columns_to_drop, axis=1, errors='ignore')

print(f"Columns retained: {df.shape[1]}")
print(f"Shape after column selection: {df.shape}")

Columns retained: 20
Shape after column selection: (4864, 20)


## Duplicate Analysis

In [4]:
# Identify duplicated hadm_id (multiple notes per admission)
duplicated_hadm = df[df.duplicated(subset='hadm_id', keep=False)].sort_values('hadm_id')
print(f"Admissions with multiple notes (hadm_id duplicates): {len(duplicated_hadm) // 2}")

if len(duplicated_hadm) > 0:
    print(f"\nExample duplicated hadm_id:")
    print(duplicated_hadm.head(6))

# Count exact duplicates (identical rows)
exact_duplicates = df[df.duplicated(keep=False)]
print(f"\nExact duplicate rows: {len(exact_duplicates)}")

if len(exact_duplicates) > 0:
    print("Dropping exact duplicates...")
    df = df.drop_duplicates()
    print(f"Shape after removing exact duplicates: {df.shape}")

Admissions with multiple notes (hadm_id duplicates): 102

Example duplicated hadm_id:
      subject_id   hadm_id            charttime            storetime  \
4644    19781816  20200492  2157-10-27 03:00:00  2157-10-29 23:12:00   
4823    19998560  20200492  2157-10-27 03:00:00  2157-10-29 23:12:00   
4802    19998539  20222315  2187-11-03 03:00:00  2187-11-05 01:15:00   
4279    19032473  20222315  2187-11-03 03:00:00  2187-11-05 01:15:00   
3758    17922874  20343031  2158-05-23 03:00:00  2158-05-23 20:55:00   
4862    19998599  20343031  2158-05-23 03:00:00  2158-05-23 20:55:00   

                                                    HPI  \
4644  :\n___ with history of DM2, hypertension, hype...   
4823  :\n___ with history of DM2, hypertension, hype...   
4802  :\n___ up F w/h/o HTN, dementia, asthma (not o...   
4279  :\n___ up F w/h/o HTN, dementia, asthma (not o...   
3758  :\n___ with H/O of type 2 diabetes mellitus, h...   
4862  :\n___ with H/O of type 2 diabetes mellitus, h...

## Missing Data Analysis

In [5]:
# Comprehensive missing data report
missing_report = pd.DataFrame({
    'Column': df.columns,
    'Non-Null Count': df.count(),
    'Null Count': df.isnull().sum(),
    'Completeness (%)': (df.count() / len(df) * 100).round(1)
}).sort_values('Completeness (%)')

print("\nMissing Data Summary:")
print(missing_report.to_string(index=False))

# Identify columns with >70% missing (likely non-informative)
sparse_cols = missing_report[missing_report['Completeness (%)'] < 30]
if len(sparse_cols) > 0:
    print(f"\n⚠️  Highly sparse columns (<30% complete):")
    print(sparse_cols[['Column', 'Completeness (%)']].to_string(index=False))


Missing Data Summary:
         Column  Non-Null Count  Null Count  Completeness (%)
            dod             398        4466               8.2
         gender            1363        3501              28.0
            age            1363        3501              28.0
chief_complaint            4852          12              99.8
      invasions            4852          12              99.8
      storetime            4864           0             100.0
     subject_id            4864           0             100.0
        hadm_id            4864           0             100.0
          X-ray            4864           0             100.0
             CT            4864           0             100.0
     Ultrasound            4864           0             100.0
           CATH            4864           0             100.0
            ECG            4864           0             100.0
      charttime            4864           0             100.0
  physical_exam            4864           0    

# Data Preparation

Data preparation ensures **consistency, coherence, and readiness** for feature engineering.

## Text Cleaning Strategy

Clinical notes contain inconsistent formatting and various representations of missing values:
- Extra whitespace (leading/trailing spaces)
- Placeholder missing values: "NA", "N/A", empty strings

### Cleaning Operations
1. **Strip whitespace** from all string fields
2. **Standardize missing values** to proper NaN
3. **Normalize case** (uppercase for code matching)

In [6]:
# Strip whitespace from all string columns
string_cols = df.select_dtypes(include='object').columns
for col in string_cols:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Standardize missing value representations
df.replace(['', ' ', 'NA', 'N/A', 'na', 'n/a'], np.nan, inplace=True)

print("✓ Text cleaning complete")
print(f"  - Stripped whitespace from {len(string_cols)} text columns")
print(f"  - Standardized missing value representations")

✓ Text cleaning complete
  - Stripped whitespace from 17 text columns
  - Standardized missing value representations


## Temporal Data Processing

In [7]:
# Convert temporal columns to datetime
for col in ['charttime', 'storetime']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

# Compute storage lag (time between event and documentation)
if 'charttime' in df.columns and 'storetime' in df.columns:
    df["store_lag_hours"] = (df["storetime"] - df["charttime"]).dt.total_seconds() / 3600

print("✓ Temporal processing complete")
if 'store_lag_hours' in df.columns:
    print(f"  - Store lag: mean={df['store_lag_hours'].mean():.1f}h, median={df['store_lag_hours'].median():.1f}h")
    print(f"  - Range: {df['store_lag_hours'].min():.1f}h to {df['store_lag_hours'].max():.1f}h")

✓ Temporal processing complete
  - Store lag: mean=760.8h, median=719.1h
  - Range: -6526.5h to 8543.6h


## Identifier & Demographic Normalization

In [8]:
# Convert identifiers to numeric
df["subject_id"] = pd.to_numeric(df["subject_id"], errors="coerce")
df["hadm_id"] = pd.to_numeric(df["hadm_id"], errors="coerce")

# Process age
if 'age' in df.columns:
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    # Remove implausible ages
    implausible_ages = df[df["age"] > 120].shape[0]
    df.loc[df["age"] > 120, "age"] = np.nan
    print(f"✓ Age normalization: removed {implausible_ages} implausible ages (>120)")

# Standardize gender
if 'gender' in df.columns:
    df["gender"] = df["gender"].str.upper().replace({
        "FEMALE": "F",
        "MALE": "M"
    })
    print(f"✓ Gender standardization: {df['gender'].value_counts().to_dict()}")

# Standardize ICD codes
if 'icd_code' in df.columns:
    df["icd_code"] = df["icd_code"].str.upper().str.strip()
    df["icd_code"] = df["icd_code"].apply(
        lambda x: re.sub(r"[^A-Z0-9.]", "", x) if isinstance(x, str) else x
    )
    print(f"✓ ICD code normalization: {df['icd_code'].nunique()} unique codes")

✓ Age normalization: removed 0 implausible ages (>120)
✓ Gender standardization: {'M': 781, 'F': 582}
✓ ICD code normalization: 20 unique codes


## Final Data Quality Check

In [9]:
print("="*70)
print("CLEANED DATASET SUMMARY")
print("="*70)
print(f"\nShape: {df.shape[0]} admissions × {df.shape[1]} columns")
print(f"Unique patients (subject_id): {df['subject_id'].nunique()}")
print(f"Unique admissions (hadm_id): {df['hadm_id'].nunique()}")
print(f"\nAdmissions per patient:")
admissions_per_patient = df.groupby('subject_id').size()
print(f"  - Mean: {admissions_per_patient.mean():.1f}")
print(f"  - Median: {admissions_per_patient.median():.0f}")
print(f"  - Max: {admissions_per_patient.max()}")

print(f"\nFinal missing data summary:")
missing_final = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing_final[missing_final > 0].head(10).round(1))

CLEANED DATASET SUMMARY

Shape: 4864 admissions × 21 columns
Unique patients (subject_id): 4392
Unique admissions (hadm_id): 4761

Admissions per patient:
  - Mean: 1.1
  - Median: 1
  - Max: 5

Final missing data summary:
dod                91.8
gender             72.0
age                72.0
store_lag_hours    66.2
storetime          60.6
charttime          60.4
invasions           0.6
chief_complaint     0.2
dtype: float64


# Feature Extraction

Feature extraction converts **unstructured clinical text and structured data** into **quantitative variables** suitable for clustering.

## Feature Extraction Functions

We define specialized functions for each clinical domain:
- **Physical examination**: Vital signs, clinical signs
- **Echocardiography**: Cardiac function, valvular disease
- **Cardiac catheterization**: Coronary artery disease, hemodynamics
- **ECG**: Rhythm, ischemia, conduction abnormalities
- **CT**: Pulmonary and aortic pathology

In [10]:
# ============================================================================
# PHYSICAL EXAMINATION FEATURE EXTRACTION
# ============================================================================

def extract_physical_exam_features(text):
    """Extract vital signs and clinical signs from physical examination text."""
    if not isinstance(text, str):
        return {}
    
    t = text.upper()
    feats = {}

    # Vital Signs
    m = re.search(r'T[=:\s]*(\d{2,3}\.?\d?)', t)
    if m:
        feats['temperature'] = float(m.group(1))

    m = re.search(r'BP[=:\s]*(\d{2,3})/(\d{2,3})', t)
    if m:
        feats['bp_systolic'] = int(m.group(1))
        feats['bp_diastolic'] = int(m.group(2))

    m = re.search(r'HR[=:\s]*(\d{2,3})', t)
    if m:
        feats['heart_rate'] = int(m.group(1))

    m = re.search(r'RR[=:\s]*(\d{1,2})', t)
    if m:
        feats['respiratory_rate'] = int(m.group(1))

    m = re.search(r'O2\s*SAT[=:\s]*(\d{2,3})%?', t)
    if not m:
        m = re.search(r'(\d{2,3})%?\s*(?:ON\s*)?(?:RA|[0-9]L)', t)
    if m:
        feats['oxygen_saturation'] = int(m.group(1))

    # Clinical abbreviations
    feats['perrl'] = 1 if 'PERRL' in t or 'PERRLA' in t else 0
    feats['nad'] = 1 if 'NAD' in t else 0
    feats['rrr_cardiac'] = 1 if 'RRR' in t else 0
    feats['ctab'] = 1 if 'CTAB' in t else 0

    # Edema
    no_edema = bool(re.search(r'NO\s+(?:C/C/)?EDEMA', t))
    has_edema = bool(re.search(r'\d\+.*EDEMA|EDEMA.*\d\+', t))
    if no_edema:
        feats['edema_present'] = 0
    elif has_edema:
        feats['edema_present'] = 1

    # Breath sounds
    feats['wheezes_present'] = 1 if re.search(r'WHEEZ', t) and not re.search(r'NO\s+WHEEZ', t) else 0
    has_crackles = bool(re.search(r'CRACKLE|RALE', t))
    no_crackles = bool(re.search(r'NO\s+(?:CRACKLE|RALE)', t))
    feats['crackles_present'] = 1 if has_crackles and not no_crackles else 0

    # Murmur
    has_murmur = bool(re.search(r'MURMUR', t))
    no_murmur = bool(re.search(r'NO\s+MURMUR', t))
    feats['murmur_present'] = 1 if has_murmur and not no_murmur else 0

    return feats

print("✓ Physical exam feature extraction function defined")

✓ Physical exam feature extraction function defined


In [11]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def parse_report_list(report_data):
    """Convert nested report list string to list of text segments."""
    if pd.isna(report_data) or report_data == '[]':
        return []
    if isinstance(report_data, str):
        report_data = report_data.strip('[]')
        texts = re.findall(r"'([^']*)'", report_data)
        return [t for t in texts if t.strip() and t != '___']
    return []

# Define clinical grading scales
severity_map = {
    'trace': 1, 'trivial': 1, 'mild': 2, 'moderate': 3,
    'moderately': 3, 'severe': 4, 'mod-severe': 3.5,
    'moderate to severe': 3.5, 'mod to severe': 3.5
}
valve_grades = {'1+': 1, '2+': 2, '3+': 3, '4+': 4}

print("✓ Helper functions defined")

✓ Helper functions defined


In [12]:
# ============================================================================
# ECHOCARDIOGRAPHY FEATURE EXTRACTION
# ============================================================================

def extract_ultrasound_features(reports, severity_map, valve_grades):
    """Extract cardiac function and valve disease from echocardiography reports."""
    feats = {}
    full_text = ' '.join(reports).upper() if reports else ''
    if not full_text.strip():
        return feats

    # LVEF extraction
    patterns = [
        r'LVEF[=:\s]*(\d{1,2})%?',
        r'EJECTION FRACTION[=:\s]*(\d{1,2})%?',
        r'EF[=:\s]*(\d{1,2})%?'
    ]
    for p in patterns:
        m = re.search(p, full_text)
        if m:
            ef = int(m.group(1))
            if 10 <= ef <= 80:
                feats['lvef'] = ef
                if ef >= 50:
                    feats['lvef_category'] = 'normal'
                elif ef >= 40:
                    feats['lvef_category'] = 'mildly_reduced'
                elif ef >= 30:
                    feats['lvef_category'] = 'moderately_reduced'
                else:
                    feats['lvef_category'] = 'severely_reduced'
                break

    # LV dilatation
    if re.search(r'LV.*DILAT|LEFT VENTRICULAR.*DILAT', full_text):
        feats['lv_dilated'] = 1
    elif re.search(r'NORMAL.*LV.*SIZE', full_text):
        feats['lv_dilated'] = 0

    # Wall motion abnormalities
    feats['wall_motion_abnormality'] = 1 if bool(re.search(r'HYPOKIN|AKINESIS|DYSKINESIS', full_text)) else 0

    # Mitral regurgitation
    mr = re.search(r'MITRAL REGURGITATION.*?(\d\+|TRACE|MILD|MODERATE|SEVERE)', full_text)
    if mr:
        sev = mr.group(1).lower()
        if sev in valve_grades:
            feats['mitral_regurgitation'] = valve_grades[sev]
        elif sev in severity_map:
            feats['mitral_regurgitation'] = severity_map[sev]

    # Aortic stenosis
    feats['aortic_stenosis'] = 1 if bool(
        re.search(r'AORTIC STENOSIS', full_text)
        and not re.search(r'NO.*AORTIC STENOSIS', full_text)
    ) else 0

    # Pulmonary hypertension
    feats['pulmonary_hypertension'] = 1 if bool(re.search(r'PULMONARY.*HYPERTENSION', full_text)) else 0

    # Pericardial effusion
    feats['pericardial_effusion'] = 1 if bool(
        re.search(r'PERICARDIAL EFFUSION', full_text)
        and not re.search(r'NO PERICARDIAL', full_text)
    ) else 0

    # RV dysfunction
    feats['rv_dysfunction'] = 1 if bool(re.search(r'RV.*DYSFUNC|RIGHT VENT.*DYSFUNC', full_text)) else 0

    return feats

print("✓ Ultrasound feature extraction function defined")

✓ Ultrasound feature extraction function defined


In [13]:
# ============================================================================
# CARDIAC CATHETERIZATION FEATURE EXTRACTION
# ============================================================================

def extract_cath_features(reports):
    """Extract coronary artery disease and hemodynamics from catheterization reports."""
    feats = {}
    full_text = ' '.join(reports).upper() if reports else ''
    if not full_text.strip():
        return feats

    # CAD / number of diseased vessels
    m = re.search(r'(\d)[\s-]*VESSEL.*DISEASE', full_text)
    if m:
        feats['num_vessels_diseased'] = int(m.group(1))
        feats['coronary_artery_disease'] = 1
    elif re.search(r'NO.*SIGNIFICANT.*CAD', full_text):
        feats['coronary_artery_disease'] = 0
        feats['num_vessels_diseased'] = 0

    # Stenosis severity
    m = re.search(r'LAD.*?(\d{2,3})%', full_text)
    if m:
        feats['lad_stenosis'] = int(m.group(1))

    m = re.search(r'RCA.*?(\d{2,3})%', full_text)
    if m:
        feats['rca_stenosis'] = int(m.group(1))

    stenoses = [feats.get('lad_stenosis', 0), feats.get('rca_stenosis', 0)]
    feats['severe_stenosis'] = int(any(s >= 70 for s in stenoses))

    # Interventions
    feats['stent_placed'] = int(bool(re.search(r'STENT|PCI', full_text)))
    feats['cabg'] = int(bool(re.search(r'CABG|BYPASS', full_text)))

    # Hemodynamics
    m = re.search(r'CARDIAC INDEX.*?(\d+\.?\d*)', full_text)
    if m:
        ci = float(m.group(1))
        feats['cardiac_index'] = ci
        feats['low_cardiac_output'] = int(ci < 2.0)

    m = re.search(r'PCWP.*?(\d{1,2})', full_text)
    if m:
        pcwp = int(m.group(1))
        feats['pcwp'] = pcwp
        feats['elevated_filling_pressure'] = int(pcwp > 18)

    return feats

print("✓ Catheterization feature extraction function defined")

✓ Catheterization feature extraction function defined


In [14]:
# ============================================================================
# ECG FEATURE EXTRACTION
# ============================================================================

def extract_ecg_features(reports):
    """Extract rhythm, conduction, and ischemia features from ECG reports."""
    feats = {}
    full_text = ' '.join(reports).upper() if reports else ''
    if not full_text.strip():
        return feats

    # Rhythm
    if re.search(r'ATRIAL FIBRILLATION|A\.?\s*FIB', full_text):
        feats['rhythm'] = 'atrial_fibrillation'
        feats['atrial_fibrillation'] = 1
    elif re.search(r'SINUS', full_text):
        feats['rhythm'] = 'sinus'
        feats['atrial_fibrillation'] = 0
    else:
        feats['atrial_fibrillation'] = 0

    # Conduction blocks
    feats['lbbb'] = 1 if bool(re.search(r'LBBB|LEFT BUNDLE.*BLOCK', full_text)) else 0
    feats['rbbb'] = 1 if bool(re.search(r'RBBB|RIGHT BUNDLE.*BLOCK', full_text)) else 0

    # Ischemia
    feats['st_elevation'] = 1 if bool(
        re.search(r'ST.*ELEVATION', full_text)
        and not re.search(r'NO.*ST.*ELEVATION', full_text)
    ) else 0
    feats['st_depression'] = 1 if bool(re.search(r'ST.*DEPRESSION', full_text)) else 0
    feats['q_waves'] = 1 if bool(re.search(r'\bQ\s*WAVE', full_text)) else 0

    # LVH
    feats['lvh'] = 1 if bool(re.search(r'LVH|LEFT VENTRICULAR HYPERTROPHY', full_text)) else 0

    return feats

print("✓ ECG feature extraction function defined")

✓ ECG feature extraction function defined


In [15]:
# ============================================================================
# CT IMAGING FEATURE EXTRACTION
# ============================================================================

def extract_ct_features(reports):
    """Extract pulmonary and aortic pathology from CT reports."""
    feats = {}
    full_text = ' '.join(reports).upper() if reports else ''
    if not full_text.strip():
        return feats

    feats['pulmonary_embolism'] = 1 if bool(
        re.search(r'PULMONARY EMBOLI|PE\b', full_text)
        and not re.search(r'NO.*PE', full_text)
    ) else 0

    feats['pleural_effusion'] = 1 if bool(re.search(r'PLEURAL EFFUSION', full_text)) else 0
    feats['pulmonary_edema'] = 1 if bool(re.search(r'PULMONARY EDEMA|PULMONARY CONGESTION', full_text)) else 0

    feats['coronary_calcification'] = 1 if bool(re.search(r'CORONARY.*CALCIF', full_text)) else 0
    feats['aortic_aneurysm'] = 1 if bool(re.search(r'AORTIC ANEURYSM', full_text)) else 0
    feats['aortic_dissection'] = 1 if bool(
        re.search(r'AORTIC DISSECTION', full_text)
        and not re.search(r'NO.*DISSECTION', full_text)
    ) else 0

    return feats

print("✓ CT feature extraction function defined")

✓ CT feature extraction function defined


In [16]:
# ============================================================================
# COMPOSITE CLINICAL SCORES
# ============================================================================

def compute_composite_scores(feats):
    """
    Calculate aggregate clinical scores from individual features.
    
    Heart Failure Severity Score (0-10):
    - Combines ejection fraction, volume overload, and hemodynamics
    - Higher score = greater HF severity
    
    CAD Severity Score (0-6):
    - Combines number of vessels and severity of stenosis
    - Higher score = greater CAD burden
    
    Risk Category:
    - Low (≤5), Medium (5-10), High (>10)
    """
    
    # Heart Failure Severity Score
    hf_score = 0
    ef_map = {'normal': 0, 'mildly_reduced': 1, 'moderately_reduced': 2, 'severely_reduced': 4}
    hf_score += ef_map.get(feats.get('lvef_category'), 0)
    hf_score += feats.get('pulmonary_edema', 0)
    hf_score += feats.get('pleural_effusion', 0)
    hf_score += feats.get('elevated_filling_pressure', 0)
    hf_score += feats.get('low_cardiac_output', 0) * 2
    hf_score += feats.get('rv_dysfunction', 0)
    feats['heart_failure_severity_score'] = hf_score

    # CAD Severity Score
    cad_score = feats.get('num_vessels_diseased', 0)
    cad_score += feats.get('severe_stenosis', 0) * 2
    cad_score += feats.get('q_waves', 0)
    feats['cad_severity_score'] = cad_score

    # Risk Category
    total = hf_score + cad_score
    if total <= 5:
        feats['risk_category'] = 'Low'
    elif total <= 10:
        feats['risk_category'] = 'Medium'
    else:
        feats['risk_category'] = 'High'

    return feats

print("✓ Composite score functions defined")

✓ Composite score functions defined


In [17]:
def extract_all_features_row(row):
    """
    Extract all clinical features from a single admission (row).
    Returns a dictionary of features for that admission.

    """
    feats = {
        "subject_id": row.get("subject_id"),
        "hadm_id": row.get("hadm_id"),
        "age": row.get("age"),
        "gender": row.get("gender"),
        "icd_code": row.get("icd_code"),
        "long_title": row.get("long_title"),
    }

    # Physical exam
    if "physical_exam" in row and pd.notna(row["physical_exam"]):
        feats.update(extract_physical_exam_features(row["physical_exam"]))

    # Ultrasound
    if "Ultrasound" in row and pd.notna(row["Ultrasound"]):
        reports = parse_report_list(row["Ultrasound"])
        if reports:
            feats.update(
                extract_ultrasound_features(reports, severity_map, valve_grades)
            )

    # CATH
    if "CATH" in row and pd.notna(row["CATH"]):
        reports = parse_report_list(row["CATH"])
        if reports:
            feats.update(extract_cath_features(reports))

    # ECG
    if "ECG" in row and pd.notna(row["ECG"]):
        reports = parse_report_list(row["ECG"])
        if reports:
            feats.update(extract_ecg_features(reports))

    # CT
    if "CT" in row and pd.notna(row["CT"]):
        reports = parse_report_list(row["CT"])
        if reports:
            feats.update(extract_ct_features(reports))

    # Composite scores
    feats = compute_composite_scores(feats)

    return feats


print("Main feature extraction function defined")

Main feature extraction function defined


In [18]:
# ============================================================================
# APPLY FEATURE EXTRACTION TO ALL ADMISSIONS
# ============================================================================

print("\n" + "="*70)
print("FEATURE EXTRACTION: ADMISSION LEVEL")
print("="*70)
print(f"\nExtracting features from {len(df)} admissions...")

features_list = []
for idx, row in df.iterrows():
    if idx % 500 == 0 and idx > 0:
        print(f"  ✓ Processed {idx}/{len(df)} admissions...")
    
    features = extract_all_features_row(row)
    features_list.append(features)

df_admission_features = pd.DataFrame(features_list)

print(f"\n✓ Features extracted")
print(f"  - Shape: {df_admission_features.shape[0]} admissions × {df_admission_features.shape[1]} features")
print(f"\nFeature categories:")
print(f"  - Demographics: age, gender")
print(f"  - Physical exam: vital signs, clinical signs")
print(f"  - Cardiac function: LVEF, dilatation, wall motion")
print(f"  - CAD severity: vessels, stenosis, prior MI")
print(f"  - Arrhythmias: AF, bundle blocks")
print(f"  - Ischemia: ST changes, Q waves")
print(f"  - Complications: edema, effusions, PE")
print(f"  - Composite scores: HF severity, CAD severity, risk category")

print(f"\nFirst few feature columns:")
print(df_admission_features.columns[:15].tolist())



FEATURE EXTRACTION: ADMISSION LEVEL

Extracting features from 4864 admissions...
  ✓ Processed 500/4864 admissions...
  ✓ Processed 1000/4864 admissions...
  ✓ Processed 1500/4864 admissions...
  ✓ Processed 2000/4864 admissions...
  ✓ Processed 2500/4864 admissions...
  ✓ Processed 3000/4864 admissions...
  ✓ Processed 3500/4864 admissions...
  ✓ Processed 4000/4864 admissions...
  ✓ Processed 4500/4864 admissions...

✓ Features extracted
  - Shape: 4864 admissions × 57 features

Feature categories:
  - Demographics: age, gender
  - Physical exam: vital signs, clinical signs
  - Cardiac function: LVEF, dilatation, wall motion
  - CAD severity: vessels, stenosis, prior MI
  - Arrhythmias: AF, bundle blocks
  - Ischemia: ST changes, Q waves
  - Complications: edema, effusions, PE
  - Composite scores: HF severity, CAD severity, risk category

First few feature columns:
['subject_id', 'hadm_id', 'age', 'gender', 'icd_code', 'long_title', 'temperature', 'bp_systolic', 'bp_diastolic', 'he

# Patient-Level Feature Aggregation

## Aggregation Strategy (Clinically Motivated)

Raw extraction creates **one feature vector per admission**. We now aggregate to **one vector per patient** using clinically appropriate aggregation functions.

### Key Principle
Different features require **different aggregation** to preserve clinical meaning:

**Numeric features (LVEF, HR, BP, O2)**: mean, min, max, std
- `mean` = baseline chronic state
- `min/max` = severity extremes
- `std` = disease stability

**Binary features (CAD, AF, edema)**: max, sum, mean
- `max` = ever diagnosed (permanent marker)
- `sum` = disease burden (number of affected admissions)
- `mean` = prevalence (% of admissions)

**Composite scores**: mean, max, last
- `mean` = chronic risk level
- `max` = worst-case severity
- `last` = current state (trajectory indicator)

In [19]:
# ============================================================================
# PATIENT-LEVEL AGGREGATION SETUP
# ============================================================================

# Define aggregation rules for each feature type
aggregation_rules = {}

# Demographics
aggregation_rules['age'] = 'max'  # Most recent age

# Numeric cardiac function features → mean, min, max, std
for feat in ['lvef', 'temperature', 'heart_rate', 'bp_systolic', 'bp_diastolic', 
             'respiratory_rate', 'oxygen_saturation', 'cardiac_index', 'pcwp']:
    if feat in df_admission_features.columns:
        aggregation_rules[feat] = ['mean', 'min', 'max', 'std']

# Binary features → max (ever had), sum (burden), mean (prevalence)
binary_feats = ['lv_dilated', 'wall_motion_abnormality', 'edema_present', 'murmur_present',
                'coronary_artery_disease', 'num_vessels_diseased', 'severe_stenosis',
                'atrial_fibrillation', 'lbbb', 'rbbb', 'st_elevation', 'st_depression', 'q_waves',
                'pulmonary_edema', 'pleural_effusion', 'pericardial_effusion',
                'pulmonary_hypertension', 'pulmonary_embolism', 'rv_dysfunction']

for feat in binary_feats:
    if feat in df_admission_features.columns:
        aggregation_rules[feat] = ['max', 'sum', 'mean']

# Composite scores → mean, max, last
for feat in ['heart_failure_severity_score', 'cad_severity_score']:
    if feat in df_admission_features.columns:
        aggregation_rules[feat] = ['mean', 'max']

print("✓ Aggregation rules defined")
print(f"  - {len(aggregation_rules)} features with aggregation rules")

✓ Aggregation rules defined
  - 31 features with aggregation rules


In [20]:
# ============================================================================
# PERFORM PATIENT-LEVEL AGGREGATION
# ============================================================================

print("\n" + "="*70)
print("PATIENT-LEVEL AGGREGATION")
print("="*70)

# Group by subject_id and aggregate
df_patient_features = df_admission_features.groupby('subject_id').agg(aggregation_rules)

# Flatten multi-level column names (e.g., ('lvef', 'mean') → 'lvef_mean')
df_patient_features.columns = ['_'.join(col).strip('_') for col in df_patient_features.columns.values]

# Reset index to make subject_id a column
df_patient_features = df_patient_features.reset_index()

print(f"\n✓ Aggregation complete")
print(f"  - Input: {len(df_admission_features)} admissions")
print(f"  - Output: {len(df_patient_features)} unique patients")
print(f"  - Features: {df_patient_features.shape[1]}")
print(f"\nPatient-level feature matrix shape: {df_patient_features.shape}")
print(f"\nFirst few columns:")
print(df_patient_features.head(3))


PATIENT-LEVEL AGGREGATION

✓ Aggregation complete
  - Input: 4864 admissions
  - Output: 4392 unique patients
  - Features: 99

Patient-level feature matrix shape: (4392, 99)

First few columns:
   subject_id  age_max  lvef_mean  lvef_min  lvef_max  lvef_std  \
0    10000980     75.0        NaN       NaN       NaN       NaN   
1    10002013      NaN        NaN       NaN       NaN       NaN   
2    10002155      NaN        NaN       NaN       NaN       NaN   

   temperature_mean  temperature_min  temperature_max  temperature_std  ...  \
0              98.1             98.1             98.1              NaN  ...   
1              99.4             99.4             99.4              NaN  ...   
2               NaN              NaN              NaN              NaN  ...   

   pulmonary_embolism_max  pulmonary_embolism_sum  pulmonary_embolism_mean  \
0                     0.0                     0.0                      0.0   
1                     0.0                     0.0             

# Data Quality & Validation

In [21]:
print("\n" + "="*70)
print("FEATURE COMPLETENESS ANALYSIS")
print("="*70)

# Calculate completeness for each feature
completeness = pd.DataFrame({
    'Feature': df_patient_features.columns,
    'Non-Null Count': df_patient_features.count(),
    'Completeness (%)': (df_patient_features.count() / len(df_patient_features) * 100).round(1)
}).sort_values('Completeness (%)', ascending=False)

print("\nTop 30 most complete features:")
print(completeness.head(30).to_string(index=False))

print(f"\nFeature completeness categories:")
print(f"  - ≥80% complete: {len(completeness[completeness['Completeness (%)'] >= 80])} features")
print(f"  - 50-80% complete: {len(completeness[(completeness['Completeness (%)'] >= 50) & (completeness['Completeness (%)'] < 80)])} features")
print(f"  - <50% complete: {len(completeness[completeness['Completeness (%)'] < 50])} features")


FEATURE COMPLETENESS ANALYSIS

Top 30 most complete features:
                          Feature  Non-Null Count  Completeness (%)
                       subject_id            4392             100.0
                   lv_dilated_sum            4392             100.0
      wall_motion_abnormality_sum            4392             100.0
                 st_elevation_sum            4392             100.0
                st_depression_sum            4392             100.0
              pulmonary_edema_sum            4392             100.0
                      q_waves_sum            4392             100.0
                         lbbb_sum            4392             100.0
                         rbbb_sum            4392             100.0
              severe_stenosis_sum            4392             100.0
             pleural_effusion_sum            4392             100.0
               rv_dysfunction_sum            4392             100.0
          cad_severity_score_mean            4392    

In [22]:
print("\n" + "="*70)
print("DESCRIPTIVE STATISTICS - KEY FEATURES")
print("="*70)

key_numeric_features = [col for col in df_patient_features.columns 
                        if any(x in col for x in ['lvef_mean', 'age_max', 'heart_rate_mean', 
                                                    'oxygen_saturation_mean', 'heart_failure_severity',
                                                    'cad_severity'])]

stats = df_patient_features[key_numeric_features].describe().T
print("\n" + stats[['count', 'mean', '50%', 'min', 'max']].round(1).to_string())

print("\n" + "="*70)
print("CATEGORICAL FEATURES DISTRIBUTION")
print("="*70)

# Gender distribution
if 'gender' in df_admission_features.columns:
    gender_dist = df_admission_features.groupby('subject_id')['gender'].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0])
    print(f"\nGender distribution:")
    print(gender_dist.value_counts())

# Risk categories
risk_by_patient = df_admission_features.groupby('subject_id')['risk_category'].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0])
print(f"\nRisk category distribution (by patient):")
print(risk_by_patient.value_counts())


DESCRIPTIVE STATISTICS - KEY FEATURES

                                    count  mean   50%   min    max
age_max                            1264.0  68.8  70.0  18.0   95.0
lvef_mean                          1371.0  43.5  45.0  10.0   80.0
heart_rate_mean                    2130.0  79.4  76.0  15.0  660.0
oxygen_saturation_mean             3561.0  97.5  97.0   0.0  994.0
heart_failure_severity_score_mean  4392.0   0.8   0.0   0.0    8.0
heart_failure_severity_score_max   4392.0   0.8   0.0   0.0    9.0
cad_severity_score_mean            4392.0   0.5   0.0   0.0    6.0
cad_severity_score_max             4392.0   0.5   0.0   0.0    6.0

CATEGORICAL FEATURES DISTRIBUTION

Gender distribution:
gender
M    731
F    533
Name: count, dtype: int64

Risk category distribution (by patient):
risk_category
Low       4303
Medium      87
High         2
Name: count, dtype: int64


Finally, we save the datasets with the features that are combined to get the patient profiles.

In [23]:
# Save patient-level features
output_file = "heart_diagnoses_features.csv"
df_patient_features.to_csv(Path().resolve().parent / 'Features' / output_file, index=False)
print(f"\n✓ Saved: {output_file}")
print(f"  - {len(df_patient_features)} unique patients")
print(f"  - {df_patient_features.shape[1]} features")


✓ Saved: heart_diagnoses_features.csv
  - 4392 unique patients
  - 99 features
